# Chapter 12: Dataset and DataLoader

[Read this chapter online](https://jackluu.io/book/section-4-training/ch12-dataset-and-dataloader/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch12-dataset-and-dataloader.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 12: Dataset and DataLoader

![You are here: Training](../assets/diagrams/ch12-where-we-are.png){ width="756" }
*Figure 12.1: We begin the training module by preparing our data pipeline.*

We have our model, and we know our goal is to predict the next token (from Chapter 11). Now we need to feed data into the engine. Instead of pushing one character at a time, we will feed the model thousands of examples simultaneously. In this chapter, we build a pipeline to prepare and batch our Shakespeare dataset.

In this chapter you will:

- Use a sliding window to generate training examples.
- Understand how batches process multiple sequences in parallel.
- Build a PyTorch Dataset and DataLoader.

**Words to Know**
    - **Block Size**: The maximum number of tokens the model can look at at one time (its context window).
    - **Batching**: Grouping multiple training examples together and processing them at the same time.
    - **Dataset**: A PyTorch class that defines how to retrieve a single training example.
    - **DataLoader**: A PyTorch utility that automatically groups individual examples into batches and shuffles them.

## Theory

### The Sliding Window

In the last chapter, we saw how a single sequence provides multiple training examples. But how do we extract these sequences from a massive text file like the complete works of Shakespeare?

We use a sliding window (Figure 12.2).

![A sliding window over text](../assets/diagrams/ch12-sliding-window.png){ width="709" }
*Figure 12.2: A sliding window creates multiple, overlapping examples from one long text.*

Imagine a window that can only see a certain number of characters at a time. This is our `block_size` (the maximum context the model can handle). We place this window at the beginning of the text to grab our first sequence. Then, we slide the window one character to the right to grab our second sequence. We repeat this until we reach the end of the text.

### Processing in Parallel: Batches

If we fed each sequence to the model one by one, training would crawl. Modern computers, especially GPUs, are fantastic at doing the same math on many pieces of data at once, and they are wasted on one sequence at a time.

How much does it matter? Rather than guess, time it:

```python
$ python src/examples/ch12_batch_speed.py
32 sequences, one at a time : 2.191 s
the same 32 as one batch    : 0.245 s
batching is 9.0x faster on this machine
```

Nine times faster, on an ordinary laptop with no GPU. The training run in Chapter 13 takes about six and a half minutes; one sequence at a time it would run closer to an hour. On a GPU, where thousands of arithmetic units sit idle waiting for work, the gap is wider still.

![Combining examples into a batch](../assets/diagrams/ch12-batching.png){ width="559" }
*Figure 12.3: A batch stacks multiple independent examples into a single block.*

We group multiple sequences together into a **batch**, as illustrated in Figure 12.3. If our batch size is 32, we pass 32 independent sequences through the model in one go. The model processes them in parallel, calculates the loss for all 32, and averages it out.

### Why 32 and Not 1, or 1,000

Speed is only half the reason to batch. The other half is the quality of the step the model takes.

Remember what the loss is for: it produces a direction to nudge the weights. With a batch of 1, that direction comes from a single stretch of Shakespeare, which might happen to be a stage direction, a run of dialogue, or a line of mostly spaces. The model would lurch after each one, correcting hard for whatever it just saw. Averaging the loss over 32 independent sequences cancels most of that noise out, so each step points somewhere closer to the truth for the text as a whole.

Why not 1,000, then? Two reasons. The whole batch has to fit in memory at once, and memory is the limit you hit first on a laptop. Beyond that, the returns fade: averaging 1,000 sequences gives a direction only slightly truer than averaging 32, while each step costs thirty times as much. The number 32 is not sacred, and you will see other books use 16 or 64. It is simply a size that is large enough to steady the direction and small enough to fit.

This brings us to the start of the "Training" stage on our map. With our data batched and ready, we can finally feed it into the model and start the training loop.

## Code

To handle this efficiently, we use two built-in PyTorch tools: `Dataset` and `DataLoader` (flow shown in Figure 12.4).

![Code flow: raw data into dataset and dataloader](../assets/diagrams/ch12-code-flow.png){ width="618" }
*Figure 12.4: The Dataset handles the sliding window, and the DataLoader stacks the examples into a batch.*

```python
class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # We need block_size + 1 tokens to form one (input, target) pair
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Input is a block of text
        x = self.data[idx     : idx + self.block_size]
        # Target is the same text, shifted one character to the right
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

train_dataset = TextDataset(train_data, gpt_cfg.block_size)
val_dataset   = TextDataset(val_data,   gpt_cfg.block_size)

# DataLoader automatically batches the data for us
train_loader = DataLoader(
    train_dataset, batch_size=train_cfg.batch_size, shuffle=True
)
```

And how to run the full script:

```python
$ python src/ch11_dataloader.py
Data loaded: 1,115,394 total tokens
  Train : 1,003,854 tokens
  Val   : 111,540 tokens

Dataset sizes:
  Train examples: 1,003,726
  Val   examples: 111,412

DataLoader config:
  Batch size  : 32
  Train batches per epoch: 31,367

--- Inspecting one batch ---
x_batch shape: torch.Size([32, 128])  (batch_size, block_size)
y_batch shape: torch.Size([32, 128])  (batch_size, block_size)

First example in batch:
  x (input)  : 'ness! serious vanity!\nMis-shapen chaos of well-seeming for...
  y (target) : 'ess! serious vanity!\nMis-shapen chaos of well-seeming form...
  (y is x shifted by 1 character)

DataLoader ready! Ready for Chapter 13.
```

**What just happened:**

1.  We loaded our text file and converted it to token IDs.
2.  Lines 12 and 14 implement the sliding window logic in `TextDataset`, returning the sequence and its shifted target starting at `idx`.
3.  Lines 21 to 23 create a `DataLoader` that automatically batches and shuffles the training data.
4.  We grabbed one batch and inspected it to confirm the target `y` is just the input `x` shifted by one character.

**Shape Check:**

Table 12.1 lists the shapes of the batched input and target.

**Table 12.1:** Tensor shapes for the batched input and target sequences.

| Variable | Shape | Meaning |
| :--- | :--- | :--- |
| `x_batch` | `[32, 128]` | 32 sequences, each containing 128 input characters. |
| `y_batch` | `[32, 128]` | 32 sequences, each containing 128 target characters. |

## Try It

We can see the sliding window in action with a tiny dataset.

```python
"""Show how a sliding window creates multiple overlapping examples."""
import torch

data = torch.tensor([10, 20, 30, 40, 50, 60, 70])
block_size = 3

print(f"Data: {data.tolist()}")
print(f"Block size: {block_size}\n")

# A simple loop to show the sliding window
for i in range(len(data) - block_size):
    x = data[i : i + block_size]
    y = data[i + 1 : i + block_size + 1]
    print(f"Example {i+1}:")
    print(f"  Input  : {x.tolist()}")
    print(f"  Target : {y.tolist()}")
```

Lines 12 and 13 slice the array to create overlapping sequences for the input and target.

```python
$ python src/examples/ch12_batching_demo.py
Data: [10, 20, 30, 40, 50, 60, 70]
Block size: 3

Example 1:
  Input  : [10, 20, 30]
  Target : [20, 30, 40]
Example 2:
  Input  : [20, 30, 40]
  Target : [30, 40, 50]
Example 3:
  Input  : [30, 40, 50]
  Target : [40, 50, 60]
Example 4:
  Input  : [40, 50, 60]
  Target : [50, 60, 70]
```

**Try It**
    Open `src/examples/ch12_batching_demo.py`. Change `block_size` to 4 and run it again. Notice how the number of available examples decreases.

**In Business**
    Imagine your house-style assistant needs to learn from a massive archive of 100,000 corporate documents. You wouldn't train it by showing it one word at a time. Processing data in parallel batches is like having the assistant review 32 different emails simultaneously, learning from all of them at once. It is the key to training efficiently at scale.

**Watch Out**
    Be careful with the `__len__` of your dataset. If you have 100 characters and a `block_size` of 10, you can only create 90 starting positions because you need 11 characters (10 for input, 1 extra for the target) for a valid example. That is why the code uses `len(self.data) - self.block_size`.

## Key Takeaways

- A sliding window extracts overlapping sequences from a continuous block of text.
- Batching processes multiple independent sequences in parallel, dramatically speeding up training.
- PyTorch's `Dataset` defines how to grab a single example.
- PyTorch's `DataLoader` handles the tedious work of grouping examples into batches and shuffling them.

## Check Your Understanding

1. If you have a sequence of 1000 tokens and a block size of 100, how many examples can a sliding window extract?
2. Why is batching important for training speed?
3. Why do we shuffle the training data?


## Further Reading

**Why the field started building bigger.** Before this, deciding how large to make a model, how much text to train it on, and how much compute to spend was guesswork. The paper measured all three and found the error falls along smooth, predictable curves across a very wide range of sizes. That turned model building into a budgeting exercise, and it is the reason the industry spent the following years scaling up. It also explains the ceiling on the model you train here: a few hundred thousand parameters and a few hundred thousand characters of Shakespeare buy a certain quality of text, and no more.

<div class="refs" markdown>

Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., Chess, B., Child, R., Gray, S., Radford, A., Wu, J., & Amodei, D. (2020). *Scaling laws for neural language models* (arXiv:2001.08361). arXiv. https://doi.org/10.48550/arXiv.2001.08361

</div>

---

### `src/ch11_dataloader.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch11_dataloader.py"   # a cell has none, and the file uses it to find the text

"""
Create the Dataset and DataLoader for training.
This file belongs to Chapter 12.
Run: python src/ch11_dataloader.py
"""
import os
import sys
import torch
from torch.utils.data import Dataset, DataLoader


from src.utils.config import GPTConfig, TrainConfig

# Settings
gpt_cfg   = GPTConfig()
train_cfg = TrainConfig()

DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

if not os.path.exists(DATA_PATH):
    print("ERROR: shakespeare.txt not found. Run download_data.py")
    sys.exit(1)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
char_to_id = {ch: i for i, ch in enumerate(chars)}

# Store IDs in a PyTorch tensor
data  = torch.tensor([char_to_id[c] for c in text], dtype=torch.long)
n     = len(data)
split = int(0.9 * n)
train_data = data[:split]
val_data   = data[split:]

# --- The Idea ---
class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # We need block_size + 1 tokens to form one (input, target) pair
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Input is a block of text
        x = self.data[idx     : idx + self.block_size]
        # Target is the same text, shifted one character to the right
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

train_dataset = TextDataset(train_data, gpt_cfg.block_size)
val_dataset   = TextDataset(val_data,   gpt_cfg.block_size)

# DataLoader automatically batches the data for us
train_loader = DataLoader(
    train_dataset, batch_size=train_cfg.batch_size, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=train_cfg.batch_size, shuffle=False
)

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 12: Dataset and DataLoader\n")

    print(f"Data loaded: {n:,} total tokens")
    print(f"  Train : {len(train_data):,} tokens")
    print(f"  Val   : {len(val_data):,} tokens")

    print(f"\nDataset sizes:")
    print(f"  Train examples: {len(train_dataset):,}")
    print(f"  Val   examples: {len(val_dataset):,}")
    print(f"\nDataLoader config:")
    print(f"  Batch size  : {train_cfg.batch_size}")
    print(f"  Train batches per epoch: {len(train_loader):,}")

    print("\n--- Inspecting one batch ---")
    x_batch, y_batch = next(iter(train_loader))
    print(f"x_batch shape: {x_batch.shape}  (batch_size, block_size)")
    print(f"y_batch shape: {y_batch.shape}  (batch_size, block_size)")

    id_to_char = {i: c for i, c in enumerate(chars)}
    decode = lambda ids: "".join([id_to_char[i.item()] for i in ids])

    print(f"\nFirst example in batch:")
    print(f"  x (input)  : {repr(decode(x_batch[0]))[:60]}...")
    print(f"  y (target) : {repr(decode(y_batch[0]))[:60]}...")
    print(f"  (y is x shifted by 1 character)")

    print("\nDataLoader ready! Ready for Chapter 13.")

---

### `src/examples/ch12_batch_speed.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch12_batch_speed.py"   # a cell has none, and the file uses it to find the text

"""
Time 32 sequences run one at a time against the same 32 run as one batch.
This file belongs to Chapter 12.
Run: python src/examples/ch12_batch_speed.py
"""
import os
import sys
import time

import torch

from src.ch09_gpt_model import GPT
from src.utils.config import GPTConfig

cfg = GPTConfig()
torch.manual_seed(42)
model = GPT(cfg)
batch = torch.randint(0, cfg.vocab_size, (32, cfg.block_size))

with torch.no_grad():
    start = time.perf_counter()
    for row in batch:                       # one sequence at a time
        model(row.unsqueeze(0))
    one_at_a_time = time.perf_counter() - start

    start = time.perf_counter()
    model(batch)                            # all 32 together
    as_one_batch = time.perf_counter() - start

print(f"32 sequences, one at a time : {one_at_a_time:.3f} s")
print(f"the same 32 as one batch    : {as_one_batch:.3f} s")
print(f"batching is {one_at_a_time / as_one_batch:.1f}x faster on this machine")

---

### `src/examples/ch12_batching_demo.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch12_batching_demo.py"   # a cell has none, and the file uses it to find the text

"""Show how a sliding window creates multiple overlapping examples."""
import torch

data = torch.tensor([10, 20, 30, 40, 50, 60, 70])
block_size = 3

print(f"Data: {data.tolist()}")
print(f"Block size: {block_size}\n")

# A simple loop to show the sliding window
for i in range(len(data) - block_size):
    x = data[i : i + block_size]
    y = data[i + 1 : i + block_size + 1]
    print(f"Example {i+1}:")
    print(f"  Input  : {x.tolist()}")
    print(f"  Target : {y.tolist()}")